Libraries

In [11]:
#install pandas
%pip install pandas
#install requests
%pip install aiohttp
#install dask
%pip install dask
%pip install --quiet dask pyarrow
#install time
%pip install time
#install datetime
%pip install datetime

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement time (from versions: none)
ERROR: No matching distribution found for time
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import time
import os
import dask.dataframe as dd
from IPython.display import display

pd.set_option('display.max_columns', None)

In [13]:
# Definir rango de fechas para Enero 2021 (no está disponible todo enero 2020)

start_date = '2021-01-01'
end_date = '2021-01-31'
# end_date = '2022-12-31'

date_range = pd.date_range(start=start_date, end=end_date)
# print(date_range)

urls = [f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv' for single_date in date_range]
df = []

In [14]:
# Importar Enero 2021 - forma 1

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()

for url in urls:
    df.append(pd.read_csv(url))

# Combinar los DataFrames diarios en uno solo.
enero = pd.concat(df, ignore_index=True)

# Calcular tiempo de carga
load_time_1 = time.time() - start_time
print(f"Tiempo de carga: {load_time_1:.2f} s")

Tiempo de carga: 1.60 s


In [15]:
# Importar Enero 2021 - forma 2 (con Dask)

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()

# Leer todos los archivos en paralelo con Dask
enero = dd.read_csv(urls, dtype={'Admin2': 'object', 'FIPS': 'object'}, assume_missing=True)

# Calcular tiempo de carga
load_time_2 = time.time() - start_time
print(f"Tiempo de carga con Dask: {load_time_2:.2f} s")

Tiempo de carga con Dask: 0.22 s


In [16]:
# 1. Cargar y visualizar los primeros 5 registros

enero.head(5)

,FIPS,Admin2,Province_State,Country_Region,Last_Update,Lat,Long_,Confirmed,Deaths,Recovered,Active,Combined_Key,Incident_Rate,Case_Fatality_Ratio
0,<NA>,<NA>,<NA>,Afghanistan,2021-01-02 05:22:33,33.93911,67.709953,52513.0,2201.0,41727.0,8585.0,Afghanistan,134.896578,4.191343
1,<NA>,<NA>,<NA>,Albania,2021-01-02 05:22:33,41.15330,20.168300,58316.0,1181.0,33634.0,23501.0,Albania,2026.409062,2.025173
2,<NA>,<NA>,<NA>,Algeria,2021-01-02 05:22:33,28.03390,1.659600,99897.0,2762.0,67395.0,29740.0,Algeria,227.809861,2.764848
3,<NA>,<NA>,<NA>,Andorra,2021-01-02 05:22:33,42.50630,1.521800,8117.0,84.0,7463.0,570.0,Andorra,10505.403482,1.034865
4,<NA>,<NA>,<NA>,Angola,2021-01-02 05:22:33,-11.20270,17.873900,17568.0,405.0,11146.0,6017.0,Angola,53.452981,2.305328


In [17]:
# 2. Mostrar el número total de filas y columnas del DataFrame.

enero_pd = enero.compute()

print('Filas en total: ', len(enero_pd))
print('Columnas en total: ', len(enero_pd.columns))

Filas en total:  124398
Columnas en total:  14


In [18]:
# 3. Describir los tipos de datos (dtypes) y convertir las columnas necesarias (por ejemplo,
# fechas).

enero.dtypes

FIPS                   string[pyarrow]
Admin2                 string[pyarrow]
Province_State         string[pyarrow]
Country_Region         string[pyarrow]
Last_Update            string[pyarrow]
Lat                            float64
Long_                          float64
Confirmed                      float64
Deaths                         float64
Recovered                      float64
Active                         float64
Combined_Key           string[pyarrow]
Incident_Rate                  float64
Case_Fatality_Ratio            float64
dtype: object

In [19]:
# 3

# Formatear la columna Last_Update a tipo datetime

enero = enero.assign(Last_Update=dd.to_datetime(enero['Last_Update'], errors='coerce'))

enero.dtypes

FIPS                   string[pyarrow]
Admin2                 string[pyarrow]
Province_State         string[pyarrow]
Country_Region         string[pyarrow]
Last_Update             datetime64[ns]
Lat                            float64
Long_                          float64
Confirmed                      float64
Deaths                         float64
Recovered                      float64
Active                         float64
Combined_Key           string[pyarrow]
Incident_Rate                  float64
Case_Fatality_Ratio            float64
dtype: object

In [20]:
# 3

# Uso de memoria antes de conversión de tipos

enero_pd = enero.compute()

# Verificar memoria usada por el DataFrame pandas resultante
memoria = enero_pd.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Memoria usada del DataFrame: {memoria:.2f} MB")

Memoria usada del DataFrame: 19.18 MB


In [21]:
# 3

# Uso de memoria después de conversión de tipos

enero['Province_State'] = enero['Province_State'].astype('category')
enero['Country_Region'] = enero['Country_Region'].astype('category')
enero['Combined_Key'] = enero['Combined_Key'].astype('category')

enero_pd = enero.compute()

# Verificar memoria usada por el DataFrame pandas resultante
memoria_optimizacion = enero_pd.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Memoria usada del DataFrame tras optimización: {memoria_optimizacion:.2f} MB")

Memoria usada del DataFrame tras optimización: 13.57 MB


In [22]:
print(f"Diferencia en uso de memoria: {(-(memoria - memoria_optimizacion) * 100 / memoria):.0f}%")

Diferencia en uso de memoria: -29%


In [23]:
# 4. Detectar y mostrar valores nulos o faltantes por columna.

enero.isnull().sum().compute()

FIPS                   23166
Admin2                 23011
Province_State          5529
Country_Region             0
Last_Update               37
Lat                     2776
Long_                   2776
Confirmed                  0
Deaths                     0
Recovered                  0
Active                     0
Combined_Key               0
Incident_Rate           2776
Case_Fatality_Ratio     1484
dtype: int64

In [24]:
# 5. Eliminar columnas irrelevantes (por ejemplo, códigos FIPS o coordenadas si no se usarán).

enero = enero.drop(columns=['FIPS', 'Admin2', 'Lat', 'Long_', 'Combined_Key'])
enero.head(0)

,Province_State,Country_Region,Last_Update,Confirmed,Deaths,Recovered,Active,Incident_Rate,Case_Fatality_Ratio


In [25]:
# 6. Estandarizar nombres de columnas (usar formato snake_case).

enero.columns = enero.columns.str.lower().str.replace(' ', '_')
enero.head(0)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio


In [26]:
# 7. Homogeneizar nombres de países (ej. “US” → “United States”).

enero['country_region'] = enero['country_region'].replace({'US': 'United States'})
enero[enero['country_region'] == 'United States'].head(1)

/usr/local/python/3.12.1/lib/python3.12/site-packages/dask/utils.py:1235: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  return getattr(__obj, self.method)(*args, **kwargs)


,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
648,Alabama,United States,2021-01-02 05:22:33,4239.0,50.0,0.0,4189.0,7587.391935,1.179523


In [27]:
# 8. Convertir la columna last_update al formato YYYY-MM-DD (día preciso)
# Asegurar datetime y mantener sólo fecha (YYYY-MM-DD)
enero['last_update'] = dd.to_datetime(enero['last_update'], errors='coerce').dt.date
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
0,<NA>,Afghanistan,2021-01-02,52513.0,2201.0,41727.0,8585.0,134.896578,4.191343


In [28]:
# 9. Crear una columna active_cases = Confirmed - Deaths - Recovered.

enero['active_cases'] = (enero['confirmed'] - enero['deaths'] - enero['recovered'])
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio,active_cases
0,<NA>,Afghanistan,2021-01-02,52513.0,2201.0,41727.0,8585.0,134.896578,4.191343,8585.0


In [29]:
# 10. Guardar el DataFrame limpio como covid_clean_enero2021.csv e indicar su tamaño en MB.

try:
    enero.compute().to_csv('covid_clean_enero2021.csv')
except:
    os.remove('covid_clean_enero2021.csv')
    enero.compute().to_csv('covid_clean_enero2021.csv')

file_size = os.path.getsize('covid_clean_enero2021.csv') / (1024 * 1024)  # Convertir a MB
print(f'El tamaño del archivo covid_clean_enero2021.csv es: {file_size:.2f} MB')

/usr/local/python/3.12.1/lib/python3.12/site-packages/dask/utils.py:1235: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  return getattr(__obj, self.method)(*args, **kwargs)
/usr/local/python/3.12.1/lib/python3.12/site-packages/dask/utils.py:1235: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  return getattr(__obj, self.method)(*args, **kwargs)
/usr/local/python/3.12.1/lib/python3.12/site-packages/dask/utils.py:1235: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be use

El tamaño del archivo covid_clean_enero2021.csv es: 12.22 MB
